# 02 – Feature Engineering

This notebook builds the features used by the predictive models in notebook 04.

**Features created:**
- Rolling team points scored / allowed (10-game window)
- Point differential per game
- True Shooting % (TS%) per player per game
- Usage Rate per player per game
- Season-aggregated per-game stats for each player

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_games, load_games_details
from src.features import (
    add_rolling_team_stats,
    add_point_differential,
    add_true_shooting,
    add_usage_rate,
    aggregate_player_season_stats,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

RAW_DIR = os.path.join(os.path.abspath('..'), 'data', 'raw')

## 1. Rolling Team Stats

In [ ]:
games = load_games(RAW_DIR)
games = add_rolling_team_stats(games, window=10)
games = add_point_differential(games)

games[['GAME_DATE_EST', 'HOME_ROLLING_PTS', 'HOME_ROLLING_PTS_ALLOWED',
       'AWAY_ROLLING_PTS', 'AWAY_ROLLING_PTS_ALLOWED', 'POINT_DIFF']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(games['HOME_ROLLING_PTS'].dropna(), bins=30, color='royalblue', alpha=0.7, label='Home')
axes[0].hist(games['AWAY_ROLLING_PTS'].dropna(), bins=30, color='tomato', alpha=0.7, label='Away')
axes[0].set_title('Rolling 10-Game Average Points')
axes[0].set_xlabel('Avg Points')
axes[0].legend()

axes[1].hist(games['POINT_DIFF'].dropna(), bins=40, color='seagreen', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Point Differential (Home − Away)')
axes[1].set_xlabel('Point Diff')

plt.tight_layout()
plt.show()

## 2. Player-Level Advanced Metrics

In [ ]:
details = load_games_details(RAW_DIR)

# Merge SEASON from games (needed for aggregation)
if 'SEASON' in games.columns:
    season_map = games.set_index('GAME_ID')['SEASON']
    details['SEASON'] = details['GAME_ID'].map(season_map)

details = add_true_shooting(details)
details = add_usage_rate(details)

details[['PLAYER_NAME', 'MIN', 'PTS', 'FGA', 'FTA', 'TS_PCT', 'USAGE_RATE']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ts_data = details['TS_PCT'].replace([np.inf, -np.inf], np.nan).dropna()
ts_data = ts_data[ts_data > 0]
axes[0].hist(ts_data, bins=40, color='orchid', alpha=0.8)
axes[0].set_title('True Shooting % Distribution')
axes[0].set_xlabel('TS%')

usage_data = details['USAGE_RATE'].replace([np.inf, -np.inf], np.nan).dropna()
usage_data = usage_data[usage_data > 0]
axes[1].hist(usage_data, bins=40, color='darkorange', alpha=0.8)
axes[1].set_title('Usage Rate Distribution')
axes[1].set_xlabel('Possessions Used per Minute')

plt.tight_layout()
plt.show()

## 3. Season-Aggregated Player Stats

In [ ]:
if 'SEASON' in details.columns:
    player_season = aggregate_player_season_stats(details)
    print(player_season.shape)
    player_season.head(10)
else:
    print('SEASON column not available — skipping aggregation')
    player_season = None

In [ ]:
if player_season is not None:
    top20 = player_season.nlargest(20, 'PTS_PG')[['PLAYER_ID', 'SEASON', 'GP', 'PTS_PG', 'REB_PG', 'AST_PG']]
    print(top20.to_string(index=False))

## 4. Save Processed Features

In [ ]:
processed_dir = os.path.join(os.path.abspath('..'), 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

games.to_csv(os.path.join(processed_dir, 'games_features.csv'), index=False)
details.to_csv(os.path.join(processed_dir, 'details_features.csv'), index=False)
if player_season is not None:
    player_season.to_csv(os.path.join(processed_dir, 'player_season_stats.csv'), index=False)

print('Saved processed features to data/processed/')